# 01 - Qualidade e integridade dos dados

## tl;dr
Este notebook valida chaves, duplicidades, valores ausentes, custos, odometro e datas antes da construcao da base mensal.

## Context & Methods

A qualidade dos dados e avaliada antes de qualquer modelagem para evitar conclusoes fortes com base em registros inconsistentes.

### Key Assumptions
- Dados brutos permanecem imutaveis.
- OS em `2026-01-01` serao excluidas da base analitica por estarem fora da janela 2020-2025.
- Custos negativos sao preservados no diagnostico e tratados com cautela na modelagem.

In [45]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
(PROJECT_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
FIGURES = REPORTS / "figures"
TABLES = REPORTS / "tables"
ANALYSIS_START = pd.Timestamp("2020-01-01")
ANALYSIS_END_EXCLUSIVE = pd.Timestamp("2026-01-01")
ANALYSIS_END = pd.Timestamp("2025-12-31")
KM_MIN_MES_ALVO = 500.0
PREVENTIVE_VMRS_CODES = {"PM"}

for path in [DATA_INTERIM, DATA_PROCESSED, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

FILES = {
    "dim_carretas": DATA_RAW / "dim_carretas_2020-01-01_to_2025-12-31.csv",
    "fato_contratos": DATA_RAW / "fato_contratos_2020-01-01_to_2025-12-31.csv",
    "fato_gps": DATA_RAW / "fato_gps_2020-01-01_to_2025-12-31.csv",
    "fato_readings": DATA_RAW / "fato_readings_2020-01-01_to_2025-12-31.csv",
    "fato_wo": DATA_RAW / "fato_wo_2020-01-01_to_2025-12-31.csv",
    "fato_wo_labour": DATA_RAW / "fato_wo_labour_2020-01-01_to_2025-12-31.csv",
    "fato_wo_parts": DATA_RAW / "fato_wo_parts_2020-01-01_to_2025-12-31.csv",
}

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def read_csv(name: str, **kwargs) -> pd.DataFrame:
    return normalize_columns(pd.read_csv(FILES[name], low_memory=False, **kwargs))

def month_start(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce").dt.to_period("M").dt.to_timestamp()

def as_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")

def mode_or_unknown(series: pd.Series, unknown: str = "SEM_INFORMACAO") -> str:
    values = series.dropna().astype(str)
    if values.empty:
        return unknown
    return values.mode().iloc[0]

## Data

### 1. Carregar chaves principais

In [46]:
dim = read_csv("dim_carretas")
wo_keys = read_csv("fato_wo", usecols=["ID_OS", "ID_CARRETA", "DATA_OS"])
labour_keys = read_csv("fato_wo_labour", usecols=["ID_LINHA_MAO_OBRA", "ID_OS", "ID_CARRETA", "DATA_OS"])
parts_keys = read_csv("fato_wo_parts", usecols=["ID_LINHA_PECA", "ID_OS", "ID_CARRETA", "DATA_OS"])
readings_keys = read_csv("fato_readings", usecols=["ID_LEITURA", "ID_CARRETA", "DATA_LEITURA", "ID_OS"])
contracts_keys = read_csv("fato_contratos", usecols=["ID_CONTRATO_CARRETA", "ID_CARRETA", "DATA_INICIO", "DATA_FIM"])
gps_keys = read_csv("fato_gps", usecols=["ID_CARRETA", "DATA_HORA_GPS"])
labour_class = read_csv("fato_wo_labour", usecols=["ID_OS", "ID_CARRETA", "DATA_OS", "COD_TIPO_SERVICO", "COD_SISTEMA_VMRS", "SISTEMA_VMRS", "CUSTO_INTERNO_MAO_OBRA"])
parts_quality = read_csv("fato_wo_parts", usecols=["ID_OS", "ID_CARRETA", "DATA_OS", "FLAG_GARANTIA", "CUSTO_INTERNO_PECA"])

datasets = {
    "dim_carretas": dim,
    "fato_wo": wo_keys,
    "fato_wo_labour": labour_keys,
    "fato_wo_parts": parts_keys,
    "fato_readings": readings_keys,
    "fato_contratos": contracts_keys,
    "fato_gps": gps_keys,
}

### 2. Validar chaves, duplicidades e integridade referencial

In [47]:
dim_ids = set(dim["id_carreta"].dropna())
wo_ids = set(wo_keys["id_os"].dropna())

primary_keys = {
    "dim_carretas": "id_carreta",
    "fato_wo": "id_os",
    "fato_wo_labour": "id_linha_mao_obra",
    "fato_wo_parts": "id_linha_peca",
    "fato_readings": "id_leitura",
    "fato_contratos": "id_contrato_carreta",
}

integrity = []
for name, df in datasets.items():
    row = {"base": name, "linhas": len(df)}
    if "id_carreta" in df.columns:
        row["ids_carreta_fora_dim"] = len(set(df["id_carreta"].dropna()) - dim_ids)
    pk = primary_keys.get(name)
    if pk:
        row["pk"] = pk
        row["pk_nulos"] = int(df[pk].isna().sum())
        row["pk_duplicadas"] = int(df[pk].duplicated(keep=False).sum())
    if name in ["fato_wo_labour", "fato_wo_parts"]:
        row["id_os_fora_fato_wo"] = len(set(df["id_os"].dropna()) - wo_ids)
    integrity.append(row)

integrity = pd.DataFrame(integrity)
integrity.to_csv(TABLES / "01_integridade_chaves.csv", index=False)
integrity

,base,linhas,ids_carreta_fora_dim,pk,pk_nulos,pk_duplicadas,id_os_fora_fato_wo
0,dim_carretas,10412,0,id_carreta,0.0,0.0,NaN
1,fato_wo,238818,0,id_os,0.0,0.0,NaN
2,fato_wo_labour,759813,0,id_linha_mao_obra,0.0,0.0,0.0
3,fato_wo_parts,738754,0,id_linha_peca,0.0,0.0,0.0
4,fato_readings,290535,0,id_leitura,0.0,0.0,NaN
5,fato_contratos,32232,0,id_contrato_carreta,0.0,0.0,NaN
6,fato_gps,227795,0,NaN,NaN,NaN,NaN


### 3. Medir valores ausentes por coluna

In [48]:
missing_rows = []
for name, path in FILES.items():
    df = pd.read_csv(path, low_memory=False)
    for col, missing in df.isna().sum().items():
        missing_rows.append({
            "base": name,
            "coluna": col,
            "nulos": int(missing),
            "pct_nulos": float(missing / len(df)) if len(df) else np.nan,
        })

missing = pd.DataFrame(missing_rows).sort_values(["pct_nulos", "nulos"], ascending=False)
missing.to_csv(TABLES / "01_valores_ausentes.csv", index=False)
missing.head(30)

,base,coluna,nulos,pct_nulos
5,dim_carretas,COD_MODELO,10356,0.994622
40,fato_readings,KM_RESET_EM,288030,0.991378
41,fato_readings,KM_RESET_PARA,288030,0.991378
15,dim_carretas,COD_GRUPO_MANUTENCAO,10264,0.985786
16,dim_carretas,GRUPO_MANUTENCAO,10264,0.985786
63,fato_wo_labour,TOTAL_TERCEIRIZADO,666217,0.876817
76,fato_wo_parts,CUSTO_MEDIO_ITEM,413635,0.559909
7,dim_carretas,DATA_ENTRADA_SERVICO,5006,0.480791
51,fato_wo,COD_PROVINCIA_ESTADO,114593,0.479834
52,fato_wo,PROVINCIA_ESTADO,114593,0.479834


## Results

### 4. Diagnosticar custos negativos, zerados e extremos

In [49]:
wo_cost = read_csv("fato_wo", usecols=["ID_OS", "ID_CARRETA", "DATA_OS", "TOTAL_INTERNO_MAO_OBRA", "TOTAL_INTERNO_PECAS"])
wo_cost["data_os"] = pd.to_datetime(wo_cost["data_os"], errors="coerce")
for col in ["total_interno_mao_obra", "total_interno_pecas"]:
    wo_cost[col] = as_number(wo_cost[col])
wo_cost["custo_total"] = wo_cost["total_interno_mao_obra"].fillna(0) + wo_cost["total_interno_pecas"].fillna(0)

cost_summary = []
for col in ["total_interno_mao_obra", "total_interno_pecas", "custo_total"]:
    s = wo_cost[col]
    cost_summary.append({
        "campo": col,
        "soma": s.sum(),
        "media": s.mean(),
        "mediana": s.median(),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "negativos": int((s < 0).sum()),
        "zeros": int((s == 0).sum()),
    })

cost_summary = pd.DataFrame(cost_summary)
cost_summary.to_csv(TABLES / "01_diagnostico_custos.csv", index=False)
cost_summary

,campo,soma,media,mediana,p95,p99,max,negativos,zeros
0,total_interno_mao_obra,41101326.80,172.103136,104.425,535.5000,1043.5790,16116.00,161,5990
1,total_interno_pecas,37896772.35,158.684740,14.030,804.5345,1921.9441,31944.85,144,66031
2,custo_total,78998099.15,330.787877,146.660,1267.7800,2640.3664,32143.57,183,654


### 5. Avaliar odometro e deltas de quilometragem

In [50]:
readings = read_csv("fato_readings")
readings["data_leitura"] = pd.to_datetime(readings["data_leitura"], errors="coerce")
for col in ["km_acumulado", "km_reset_em", "km_reset_para"]:
    readings[col] = as_number(readings[col])

readings = readings.sort_values(["id_carreta", "data_leitura", "id_leitura"])
readings["km_anterior"] = readings.groupby("id_carreta")["km_acumulado"].shift(1)
readings["delta_km_bruto"] = readings["km_acumulado"] - readings["km_anterior"]
positive_delta = readings.loc[readings["delta_km_bruto"] >= 0, "delta_km_bruto"]
limite_delta_km = min(max(float(positive_delta.quantile(0.999)), 50000.0), 250000.0)
readings["delta_km_tratado"] = readings["delta_km_bruto"].where(
    (readings["delta_km_bruto"] >= 0) & (readings["delta_km_bruto"] <= limite_delta_km)
)

odo_summary = pd.DataFrame([
    {"metrica": "leituras", "valor": len(readings)},
    {"metrica": "carretas", "valor": readings["id_carreta"].nunique()},
    {"metrica": "km_acumulado_nulo", "valor": readings["km_acumulado"].isna().sum()},
    {"metrica": "km_acumulado_negativo", "valor": (readings["km_acumulado"] < 0).sum()},
    {"metrica": "leituras_com_reset", "valor": readings[["km_reset_em", "km_reset_para"]].notna().any(axis=1).sum()},
    {"metrica": "delta_km_negativo", "valor": (readings["delta_km_bruto"] < 0).sum()},
    {"metrica": "delta_km_acima_limite", "valor": (readings["delta_km_bruto"] > limite_delta_km).sum()},
    {"metrica": "limite_delta_km_usado", "valor": limite_delta_km},
])

odo_summary.to_csv(TABLES / "01_diagnostico_odometro.csv", index=False)
odo_summary

,metrica,valor
0,leituras,290535.000
1,carretas,10412.000
2,km_acumulado_nulo,0.000
3,km_acumulado_negativo,0.000
4,leituras_com_reset,2505.000
5,delta_km_negativo,2443.000
6,delta_km_acima_limite,278.000
7,limite_delta_km_usado,109592.397


### 6. Verificar consistencia temporal

In [51]:
date_checks = []
for name, df in {
    "fato_wo": wo_keys,
    "fato_wo_labour": labour_keys,
    "fato_wo_parts": parts_keys,
    "fato_readings": readings_keys,
    "fato_contratos": contracts_keys,
    "fato_gps": gps_keys,
}.items():
    for col in [c for c in df.columns if c.startswith("data_")]:
        parsed = pd.to_datetime(df[col], errors="coerce")
        date_checks.append({
            "base": name,
            "campo": col,
            "min": parsed.min(),
            "max": parsed.max(),
            "nulos": int(parsed.isna().sum()),
            "antes_2020": int((parsed < pd.Timestamp("2020-01-01")).sum()),
            "apos_2025": int((parsed >= pd.Timestamp("2026-01-01")).sum()),
        })

date_checks = pd.DataFrame(date_checks)
date_checks.to_csv(TABLES / "01_consistencia_temporal.csv", index=False)
date_checks

,base,campo,min,max,nulos,antes_2020,apos_2025
0,fato_wo,data_os,2020-01-01 09:15:10,2026-01-01 02:44:00,0,0,3
1,fato_wo_labour,data_os,2020-01-01 09:15:10,2026-01-01 02:44:00,0,0,5
2,fato_wo_parts,data_os,2020-01-01 12:31:00,2026-01-01 00:23:00,0,0,3
3,fato_readings,data_leitura,2020-01-01 07:00:23,2025-12-31 21:23:54,0,0,0
4,fato_contratos,data_inicio,2005-05-12 00:00:00,2026-04-14 13:43:00,2,13792,1228
5,fato_contratos,data_fim,2015-01-01 09:19:33,2026-03-31 22:14:00,9895,7404,1013
6,fato_gps,data_hora_gps,2025-09-30 21:01:32,2025-12-31 23:58:35,0,0,0


### 7. Verificar classificacao de manutencao preventiva, garantia e contrato

In [52]:
labour_class["data_os"] = pd.to_datetime(labour_class["data_os"], errors="coerce")
labour_class["custo_interno_mao_obra"] = as_number(labour_class["custo_interno_mao_obra"]).fillna(0)
labour_class["flag_linha_preventiva"] = (
    labour_class["cod_sistema_vmrs"].astype(str).str.upper().isin(PREVENTIVE_VMRS_CODES)
    | labour_class["sistema_vmrs"].astype(str).str.upper().str.contains("PREVENTIVE", na=False)
)
preventive_os_ids = set(labour_class.loc[labour_class["flag_linha_preventiva"], "id_os"].dropna())

parts_quality["data_os"] = pd.to_datetime(parts_quality["data_os"], errors="coerce")
parts_quality["custo_interno_peca"] = as_number(parts_quality["custo_interno_peca"]).fillna(0)
parts_quality["flag_garantia_bool"] = parts_quality["flag_garantia"].astype(str).str.upper().eq("Y")

contracts_profile = read_csv("fato_contratos", usecols=["ID_CARRETA", "TIPO_MANUTENCAO", "DATA_INICIO", "DATA_FIM"])
contracts_profile["data_inicio"] = pd.to_datetime(contracts_profile["data_inicio"], errors="coerce")
contracts_profile["data_fim"] = pd.to_datetime(contracts_profile["data_fim"], errors="coerce")
contracts_profile["duracao_contrato_meses_observada"] = (
    contracts_profile["data_fim"].fillna(ANALYSIS_END) - contracts_profile["data_inicio"]
).dt.days / 30.4375

preventive_profile = pd.DataFrame([
    {"metrica": "linhas_mao_obra", "valor": len(labour_class)},
    {"metrica": "linhas_mao_obra_preventiva", "valor": int(labour_class["flag_linha_preventiva"].sum())},
    {"metrica": "os_com_linha_preventiva", "valor": len(preventive_os_ids)},
    {"metrica": "share_linhas_preventivas", "valor": float(labour_class["flag_linha_preventiva"].mean())},
    {"metrica": "linhas_peca_garantia", "valor": int(parts_quality["flag_garantia_bool"].sum())},
    {"metrica": "share_linhas_peca_garantia", "valor": float(parts_quality["flag_garantia_bool"].mean())},
    {"metrica": "duracao_contrato_meses_mediana", "valor": float(contracts_profile["duracao_contrato_meses_observada"].median())},
])
preventive_profile.to_csv(TABLES / "01_perfil_preventiva_garantia_contrato.csv", index=False)
preventive_profile

,metrica,valor
0,linhas_mao_obra,759813.000000
1,linhas_mao_obra_preventiva,127684.000000
2,os_com_linha_preventiva,104242.000000
3,share_linhas_preventivas,0.168047
4,linhas_peca_garantia,151.000000
5,share_linhas_peca_garantia,0.000204
6,duracao_contrato_meses_mediana,15.211499


### 8. Consolidar regras para a base analitica

In [53]:
treatment_rules = pd.DataFrame([
    {"tema": "janela temporal", "regra": "manter eventos de OS com DATA_OS < 2026-01-01"},
    {"tema": "odometro", "regra": "usar delta entre leituras consecutivas, remover deltas invalidos e prorratear KM entre meses-calendario"},
    {"tema": "piso de quilometragem", "regra": f"calcular alvos por km apenas para meses com km_rodado_mes >= {KM_MIN_MES_ALVO:.0f}"},
    {"tema": "alvo preventivo", "regra": "classificar OS preventiva por linha de mao de obra VMRS PM/PREVENTIVE; pecas entram como preventivas quando a OS tem linha preventiva"},
    {"tema": "zero manutencao", "regra": "manter meses com km valido e custo preventivo zero, pois representam custo esperado mensal para orcamento"},
    {"tema": "custos negativos", "regra": "preservar no diagnostico; excluir alvo negativo apenas na modelagem preditiva"},
    {"tema": "tipo_manutencao", "regra": "usar MAINT como populacao principal da modelagem e tratar NET/MIX como segmentos/caveats"},
    {"tema": "contratos", "regra": "ligar por ID_CARRETA e mes vigente dentro de DATA_INICIO e DATA_FIM"},
])
treatment_rules.to_csv(TABLES / "01_regras_tratamento.csv", index=False)
treatment_rules

,tema,regra
0,janela temporal,manter eventos de OS com DATA_OS < 2026-01-01
1,odometro,"usar delta entre leituras consecutivas, remove..."
2,piso de quilometragem,calcular alvos por km apenas para meses com km...
3,alvo preventivo,classificar OS preventiva por linha de mao de ...
4,zero manutencao,manter meses com km valido e custo preventivo ...
5,custos negativos,preservar no diagnostico; excluir alvo negativ...
6,tipo_manutencao,usar MAINT como populacao principal da modelag...
7,contratos,ligar por ID_CARRETA e mes vigente dentro de D...


## Takeaways

- A integridade por `ID_CARRETA` deve ser validada antes de qualquer juncao.
- Deltas de odometro negativos ou anormais serao removidos e a quilometragem sera prorrateada entre meses-calendario.
- A classificacao preventiva sera construida por VMRS PM/PREVENTIVE e documentada como aproximacao operacional.
- A base mensal deve registrar filtros aplicados, preservando os dados brutos para auditoria.